# Market Prices
Visualise and analyse day-ahead Elspot prices (FI, SE2) and FCR-N capacity prices.

**Requires:**
- `FINGRID_API_KEY` for FCR-N prices
- `ENTSOE_API_KEY` for Elspot prices (FI and SE2) — falls back to synthetic if not set

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

import os
import time
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data_ingestion.fingrid import FingridClient
from src.data_ingestion.nordpool import NordPoolClient, SWEDEN_SE2_AREA_CODE

fingrid = FingridClient()
nordpool = NordPoolClient()

print('Fingrid key:', bool(fingrid.api_key))
print('ENTSO-E key:', bool(nordpool.api_key))

In [ ]:
# ── Date range ────────────────────────────────────────────────────────────────
# Elspot prices are published D-1 so the latest available is today (EET/EEST).
# Use a recent historical window to guarantee data on both sides.
END   = datetime.now(tz=timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
START = END - timedelta(days=30)

## Elspot day-ahead prices — FI and SE2

In [ ]:
from src.utils.time_utils import synthetic_price_series

idx = pd.date_range(START, END, freq='h', inclusive='left', tz='UTC')

if nordpool.api_key:
    try:
        fi_prices = nordpool.get_day_ahead_prices(START, END)
        print(f'FI Elspot: {len(fi_prices)} hours fetched from ENTSO-E')
    except Exception as e:
        print(f'ENTSO-E FI fetch failed: {e} — using synthetic')
        fi_prices = synthetic_price_series(idx)

    time.sleep(2)
    try:
        se2_prices = nordpool.get_day_ahead_prices(START, END, area=SWEDEN_SE2_AREA_CODE)
        print(f'SE2 Elspot: {len(se2_prices)} hours fetched from ENTSO-E')
    except Exception as e:
        print(f'ENTSO-E SE2 fetch failed: {e} — using synthetic')
        se2_prices = synthetic_price_series(idx) - 2  # SE2 historically ~2 EUR below FI
else:
    print('ENTSOE_API_KEY not set — using synthetic prices (register at transparency.entsoe.eu)')
    fi_prices  = synthetic_price_series(idx)
    se2_prices = fi_prices - 2

In [ ]:
spread = (fi_prices - se2_prices).reindex(fi_prices.index)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Day-ahead price (EUR/MWh)', 'FI – SE2 spread (EUR/MWh)'),
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=fi_prices.index,  y=fi_prices.values,
                         name='FI',  line=dict(color='royalblue')), row=1, col=1)
fig.add_trace(go.Scatter(x=se2_prices.index, y=se2_prices.values,
                         name='SE2', line=dict(color='seagreen')), row=1, col=1)

fig.add_trace(go.Bar(x=spread.index, y=spread.values,
                     name='FI–SE2', marker_color=spread.apply(
                         lambda v: 'tomato' if v > 0 else 'steelblue')),
              row=2, col=1)
fig.add_hline(y=0, row=2, col=1, line_dash='dash', line_color='gray')

fig.update_layout(title='Elspot day-ahead prices and FI–SE2 congestion spread',
                  hovermode='x unified', height=550)
fig.show()

## FCR-N capacity prices

In [ ]:
time.sleep(2)
fcr_n = fingrid.get_fcr_n_prices(START, END)

if len(fcr_n):
    print(f'FCR-N: {len(fcr_n)} hours  |  mean {fcr_n.mean():.2f}  |  max {fcr_n.max():.2f} EUR/MW/h')
else:
    print('No FCR-N data returned for this window.')

In [ ]:
if len(fcr_n):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fcr_n.index, y=fcr_n.values,
                             mode='lines', name='FCR-N', line=dict(color='darkorange')))
    fig.update_layout(title='FCR-N hourly capacity prices (EUR/MW/h)',
                      xaxis_title='UTC', yaxis_title='EUR/MW/h',
                      hovermode='x unified', height=350)
    fig.show()

## Price statistics and correlation

In [ ]:
frames = {'FI (EUR/MWh)': fi_prices, 'SE2 (EUR/MWh)': se2_prices}
if len(fcr_n):
    frames['FCR-N (EUR/MW/h)'] = fcr_n

stats = pd.DataFrame({k: v.describe() for k, v in frames.items()}).round(2)
stats

In [ ]:
# Hourly price distributions
fig = go.Figure()
for label, series, color in [
    ('FI', fi_prices, 'royalblue'),
    ('SE2', se2_prices, 'seagreen'),
]:
    fig.add_trace(go.Histogram(x=series.values, name=label, opacity=0.65,
                               nbinsx=40, marker_color=color))

fig.update_layout(title='Price distribution (EUR/MWh)',
                  barmode='overlay', xaxis_title='EUR/MWh',
                  yaxis_title='Count', height=350)
fig.show()

In [ ]:
# Hour-of-day price profile
fi_aligned = fi_prices.reindex(idx, method='nearest', tolerance='30min')
se2_aligned = se2_prices.reindex(idx, method='nearest', tolerance='30min')

fi_hod  = fi_aligned.groupby(fi_aligned.index.hour).mean()
se2_hod = se2_aligned.groupby(se2_aligned.index.hour).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=fi_hod.index, y=fi_hod.values,
                         mode='lines+markers', name='FI', line=dict(color='royalblue')))
fig.add_trace(go.Scatter(x=se2_hod.index, y=se2_hod.values,
                         mode='lines+markers', name='SE2', line=dict(color='seagreen')))
fig.update_layout(title='Average price by hour of day (UTC)',
                  xaxis_title='Hour (UTC)', yaxis_title='EUR/MWh',
                  xaxis=dict(tickmode='linear', dtick=2), height=350)
fig.show()

In [ ]:
# FI vs SE2 scatter — visualise congestion
common_idx = fi_prices.index.intersection(se2_prices.index)
if len(common_idx) > 1:
    fi_c  = fi_prices.reindex(common_idx)
    se2_c = se2_prices.reindex(common_idx)
    corr = fi_c.corr(se2_c)

    fig = go.Figure(go.Scatter(x=fi_c.values, y=se2_c.values,
                               mode='markers', marker=dict(size=4, opacity=0.5, color='slategray')))
    lo, hi = min(fi_c.min(), se2_c.min()), max(fi_c.max(), se2_c.max())
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode='lines',
                             line=dict(dash='dash', color='red'), name='FI = SE2'))
    fig.update_layout(title=f'FI vs SE2 price (Pearson r = {corr:.3f})',
                      xaxis_title='FI (EUR/MWh)', yaxis_title='SE2 (EUR/MWh)', height=400)
    fig.show()
    print(f'Pearson correlation FI–SE2: {corr:.3f}')